### Determining the optimal number of hidden layers and neurons for an Artificial Neural Network (ANN) 
This can be challenging and often requires experimentation. However, there are some guidelines and methods that can help you in making an informed decision:

- Start Simple: Begin with a simple architecture and gradually increase complexity if needed.
- Grid Search/Random Search: Use grid search or random search to try different architectures.
- Cross-Validation: Use cross-validation to evaluate the performance of different architectures.
- Heuristics and Rules of Thumb: Some heuristics and empirical rules can provide starting points, such as:
  -    The number of neurons in the hidden layer should be between the size of the input layer and the size of the output layer.
  -  A common practice is to start with 1-2 hidden layers.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import pickle

/Users/sandeep/Documents/Gen-AI/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
data=pd.read_csv('Churn_Modelling.csv')
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

onehot_encoder_geo = OneHotEncoder(handle_unknown='ignore')
geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)

X = data.drop('Exited', axis=1)
y = data['Exited']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Save encoders and scaler for later use
with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)

with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

In [3]:
## Define a function to create the model and try different parameters(KerasClassifier)

def create_model(neurons=32,layers=1):
    model=Sequential()
    model.add(Dense(neurons,activation='relu',input_shape=(X_train.shape[1],)))

    for _ in range(layers-1):
        model.add(Dense(neurons,activation='relu'))

    model.add(Dense(1,activation='sigmoid'))
    model.compile(optimizer='adam',loss="binary_crossentropy",metrics=['accuracy'])

    return model



In [4]:
## Create a Keras classifier
model=KerasClassifier(layers=1,neurons=32,build_fn=create_model,verbose=1)

In [5]:

# Define the grid search parameters
param_grid = {
    'neurons': [16, 32, 64, 128],
    'layers': [1, 2],
    'epochs': [50, 100]
}

In [7]:
# Perform grid search
grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=3,verbose=1)
grid_result = grid.fit(X_train, y_train)

# Print the best parameters
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))

Fitting 3 folds for each of 16 candidates, totalling 48 fits


/Users/sandeep/Documents/Gen-AI/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/sandeep/Documents/Gen-AI/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/sandeep/Documents/Gen-AI/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/sandeep/Documents/Gen-AI/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, current

Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50


2025-03-20 15:56:57.905982: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
2025-03-20 15:56:57.932875: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
2025-03-20 15:56:57.978453: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
2025-03-20 15:56:57.982554: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
2025-03-20 15:56:57.995254: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
2025-03-20 15:56:57.995839: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
2025-03-20 15:56:58.012929: I tensorflow/core/grappler/optimizers/cust

167/167 [==============================] - ETA: 0s - loss: 0.6155 - accuracy: 0.6954Epoch 2/50
Epoch 2/50
167/167 [==============================] - 4s 15ms/step - loss: 0.5530 - accuracy: 0.7326
Epoch 2/50
167/167 [==============================] - 5s 14ms/step - loss: 0.6052 - accuracy: 0.6910
Epoch 2/50
167/167 [==============================] - 5s 15ms/step - loss: 0.7385 - accuracy: 0.6015
Epoch 2/50
167/167 [==============================] - 2s 13ms/step - loss: 0.4436 - accuracy: 0.8069
Epoch 3/50
167/167 [==============================] - 2s 14ms/step - loss: 0.4624 - accuracy: 0.8061
Epoch 3/50
167/167 [==============================] - 2s 14ms/step - loss: 0.4461 - accuracy: 0.8091
Epoch 3/50
167/167 [==============================] - 2s 14ms/step - loss: 0.4513 - accuracy: 0.8035
Epoch 3/50
167/167 [==============================] - 2s 14ms/step - loss: 0.4341 - accuracy: 0.8069
Epoch 4/50
167/167 [==============================] - 2s 15ms/step - loss: 0.4318 - accuracy: 0.8

...

/Users/sandeep/Documents/Gen-AI/.venv/lib/python3.9/site-packages/scikeras/wrappers.py:289: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  warnings.warn(


  4/167 [..............................] - ETA: 2s - loss: 0.6580 - accuracy: 0.6328  

/Users/sandeep/Documents/Gen-AI/.venv/lib/python3.9/site-packages/scikeras/wrappers.py:289: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  warnings.warn(


167/167 [==============================] - 4s 18ms/step - loss: 0.5255 - accuracy: 0.7458
Epoch 2/50
167/167 [==============================] - 4s 17ms/step - loss: 0.4936 - accuracy: 0.7763
Epoch 2/50
167/167 [==============================] - 4s 17ms/step - loss: 0.4940 - accuracy: 0.7810
Epoch 2/50
167/167 [==============================] - 4s 16ms/step - loss: 0.5760 - accuracy: 0.7304
Epoch 2/50
167/167 [==============================] - 4s 19ms/step - loss: 0.4995 - accuracy: 0.7696
Epoch 2/50
167/167 [==============================] - 2s 15ms/step - loss: 0.4366 - accuracy: 0.8048
Epoch 3/50
167/167 [==============================] - 5s 16ms/step - loss: 0.5345 - accuracy: 0.7521
Epoch 2/50
167/167 [==============================] - 2s 14ms/step - loss: 0.4345 - accuracy: 0.8076
Epoch 3/50
167/167 [==============================] - 3s 15ms/step - loss: 0.4451 - accuracy: 0.8095
Epoch 3/50
167/167 [==============================] - 2s 14ms/step - loss: 0.4393 - accuracy: 0.8037
E

KeyboardInterrupt: 